# AmsiNet - Step 1: Get the Bharat Scene Text Dataset (BSTD) and build the page > block > line > word hierarchy

**Runtime:** Colab Pro, CPU is enough for this notebook (GPU is needed from Step 2).

What this notebook does
1. Mounts Google Drive and downloads BSTD detection data (about 17 GB zip) from the dataset authors' Google Drive link.
2. Reads the annotations (`BSTD_v17.57.json`: polygon, text, `script_language` per word).
3. Maps the 12 language tags to scripts (Hindi/Marathi share Devanagari, Assamese/Bengali share one script).
4. Builds a geometry-derived hierarchy (page > block > line > word) and a validation split made by image.
5. Prints dataset statistics you can use in the paper (mixed-script rates etc.) and saves an index file to Drive.

Dataset: Bharat Scene Text Dataset, https://github.com/Bhashini-IITJ/BharatSceneTextDataset (images CC BY-SA 4.0).

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT  = '/content/drive/MyDrive/amsinet'     # outputs persist here
LOCAL = '/content/bstd'                      # big files live on the fast local disk
os.makedirs(f'{ROOT}/data', exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)
!pip -q install gdown

## 2. Download BSTD detection data
If the direct download fails with a "quota exceeded" message (common for large Google Drive files), open the link from the dataset page in your browser, use **Add shortcut to Drive / Make a copy**, then set `DET_ZIP` below to the copied file in `/content/drive/MyDrive/...` and skip the download line.

In [ ]:
import gdown, glob, subprocess

DET_ID = '1S7KUYfB-lQvbu6GtvZxxDPOD5ZZ080M0'      # detection.zip (about 17 GB), from the BSTD GitHub page
DET_ZIP = f'{LOCAL}/detection.zip'

if not os.path.exists(DET_ZIP):
    gdown.download(id=DET_ID, output=DET_ZIP, quiet=False)

# unzip once (takes several minutes)
if not glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True):
    subprocess.run(['unzip', '-q', '-o', DET_ZIP, '-d', LOCAL], check=True)

ANN_PATH = glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True)[0]
IMG_ROOT = os.path.dirname(ANN_PATH)
print('annotations:', ANN_PATH)
print('image root :', IMG_ROOT)

## 3. Inspect the raw annotation values (always check before trusting label names)

In [ ]:
import json
from collections import Counter

with open(ANN_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print('images in annotation file:', len(raw))
tags = Counter()
splits = Counter()
for k, v in raw.items():
    splits[v.get('split')] += 1
    for p in v.get('annotations', {}).values():
        tags[str(p.get('script_language')).strip().lower()] += 1
print('split values:', dict(splits))
print('script_language values:')
for t, n in tags.most_common():
    print(f'  {t!r:20} {n}')

## 4. Label mapping and hierarchy builder
`LANG2SCRIPT` merges languages that share a script (the BSTD authors do the same for their end-to-end evaluation). Words with other tags (for example Urdu or Meitei, which BSTD marks as "Others") are kept in the index with script `None` and ignored in training and statistics.

In [ ]:
import numpy as np, random

LANG2SCRIPT = {
    'english': 'Latin',
    'hindi': 'Devanagari', 'marathi': 'Devanagari',
    'bengali': 'Bengali-Assamese', 'assamese': 'Bengali-Assamese',
    'gujarati': 'Gujarati', 'kannada': 'Kannada', 'malayalam': 'Malayalam',
    'odia': 'Odia', 'oriya': 'Odia', 'punjabi': 'Gurmukhi', 'panjabi': 'Gurmukhi',
    'tamil': 'Tamil', 'telugu': 'Telugu',
}
SCRIPTS = sorted(set(LANG2SCRIPT.values()))
LANGS = sorted(set(k for k in LANG2SCRIPT if k not in ('oriya', 'panjabi')))
print('script classes (%d):' % len(SCRIPTS), SCRIPTS)

def norm_lang(x):
    return str(x).strip().lower()

def poly_to_bbox(coords):
    xs = [p[0] for p in coords]; ys = [p[1] for p in coords]
    return [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))]   # x1, y1, x2, y2

def _union(boxes):
    b = np.array(boxes, dtype=float)
    return [b[:, 0].min(), b[:, 1].min(), b[:, 2].max(), b[:, 3].max()]

def group_lines(words):
    # Greedy line grouping: a word joins a line if it is vertically aligned and horizontally close.
    # Heuristic for roughly horizontal text; rotated text ends up in its own groups.
    order = sorted(range(len(words)), key=lambda i: (words[i]['bbox'][1] + words[i]['bbox'][3]) / 2)
    lines = []                                                  # each: {'ids': [...], 'box': [...]}
    for i in order:
        x1, y1, x2, y2 = words[i]['bbox']
        h = max(y2 - y1, 1.0); yc = (y1 + y2) / 2
        best, best_gap = None, None
        for li, ln in enumerate(lines):
            lx1, ly1, lx2, ly2 = ln['box']
            lh = max(ly2 - ly1, 1.0); lyc = (ly1 + ly2) / 2
            ref = max(h, lh)
            if abs(yc - lyc) > 0.5 * ref:
                continue
            gap = max(x1 - lx2, lx1 - x2, 0.0)
            if gap <= 2.5 * ref and (best is None or gap < best_gap):
                best, best_gap = li, gap
        if best is None:
            lines.append({'ids': [i], 'box': [x1, y1, x2, y2]})
        else:
            lines[best]['ids'].append(i)
            lines[best]['box'] = _union([lines[best]['box'], [x1, y1, x2, y2]])
    return lines

def box_gap(a, b):
    dx = max(a[0] - b[2], b[0] - a[2], 0.0)
    dy = max(a[1] - b[3], b[1] - a[3], 0.0)
    return (dx ** 2 + dy ** 2) ** 0.5

def group_blocks(lines, factor=1.5):
    # Merge lines whose boxes are closer than factor x median line height (union-find).
    n = len(lines)
    if n == 0:
        return []
    heights = [max(l['box'][3] - l['box'][1], 1.0) for l in lines]
    thr = factor * float(np.median(heights))
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for i in range(n):
        for j in range(i + 1, n):
            if box_gap(lines[i]['box'], lines[j]['box']) <= thr:
                parent[find(i)] = find(j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())                                # list of lists of line indices

def build_index(raw, val_frac=0.10, seed=42):
    rng = random.Random(seed)
    train_keys = [k for k, v in raw.items() if 'train' in str(v.get('split')).lower()]
    val_keys = set(rng.sample(train_keys, int(round(val_frac * len(train_keys)))))   # split by IMAGE, never by word
    index = []
    for key, rec in raw.items():
        words = []
        for p in rec.get('annotations', {}).values():
            coords = p.get('coordinates')
            if not coords:
                continue
            lang = norm_lang(p.get('script_language'))
            words.append({'bbox': poly_to_bbox(coords), 'text': p.get('text', ''),
                          'lang': lang, 'script': LANG2SCRIPT.get(lang)})
        if not words:
            continue
        lines = group_lines(words)
        blocks = group_blocks(lines)
        for li, ln in enumerate(lines):
            for wi in ln['ids']:
                words[wi]['line'] = li
        for bi, line_ids in enumerate(blocks):
            for li in line_ids:
                for wi in lines[li]['ids']:
                    words[wi]['block'] = bi
        split = 'val' if key in val_keys else ('train' if 'train' in str(rec.get('split')).lower() else 'test')
        index.append({'id': key, 'image': rec.get('image_name'), 'split': split,
                      'n_lines': len(lines), 'n_blocks': len(blocks), 'words': words})
    return index

def soft_label(words):
    # Distribution of scripts over the words of a region = coarse-level target y_l(r) from Section 3.1
    c = Counter(w['script'] for w in words if w['script'])
    tot = sum(c.values())
    return {s: c[s] / tot for s in c} if tot else {}

In [ ]:
index = build_index(raw)
print('images in index:', len(index))
print(Counter(r['split'] for r in index))

## 5. Dataset statistics for the paper (copy these numbers into Section 4 of the manuscript)

In [ ]:
import statistics as st

def stats(index):
    out = {}
    for sp in ('train', 'val', 'test', 'all'):
        recs = [r for r in index if sp == 'all' or r['split'] == sp]
        ws = [w for r in recs for w in r['words'] if w['script']]
        per_img = [len({w['script'] for w in r['words'] if w['script']}) for r in recs]
        mixed = [n >= 2 for n in per_img if n >= 1]
        # words whose script differs from the image's dominant script
        minority = tot = 0
        for r in recs:
            c = Counter(w['script'] for w in r['words'] if w['script'])
            if not c: continue
            dom = c.most_common(1)[0][0]
            tot += sum(c.values()); minority += sum(v for s, v in c.items() if s != dom)
        out[sp] = {
            'images': len(recs), 'words_labelled': len(ws),
            'words_per_script': dict(Counter(w['script'] for w in ws)),
            'words_per_language': dict(Counter(w['lang'] for w in ws)),
            'images_with_2plus_scripts_pct': round(100 * sum(mixed) / max(len(mixed), 1), 1),
            'mean_scripts_per_image': round(st.mean(per_img), 2) if per_img else 0,
            'words_not_in_dominant_script_pct': round(100 * minority / max(tot, 1), 1),
            'mean_words_per_image': round(len(ws) / max(len(recs), 1), 1),
            'mean_lines_per_image': round(st.mean(r['n_lines'] for r in recs), 1) if recs else 0,
            'mean_blocks_per_image': round(st.mean(r['n_blocks'] for r in recs), 1) if recs else 0,
        }
    return out

S = stats(index)
for sp in ('train', 'val', 'test', 'all'):
    print('=' * 8, sp)
    for k, v in S[sp].items():
        print(f'  {k}: {v}')

## 6. Save the index to Drive and look at one mixed-script example

In [ ]:
with open(f'{ROOT}/data/bstd_index.json', 'w', encoding='utf-8') as f:
    json.dump({'img_root': IMG_ROOT, 'scripts': SCRIPTS, 'index': index}, f, ensure_ascii=False)
with open(f'{ROOT}/data/bstd_stats.json', 'w', encoding='utf-8') as f:
    json.dump(S, f, indent=2, ensure_ascii=False)
print('saved to', f'{ROOT}/data')

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

mixed = [r for r in index if len({w['script'] for w in r['words'] if w['script']}) >= 3]
rec = random.Random(0).choice(mixed if mixed else index)
img = Image.open(os.path.join(IMG_ROOT, rec['image'])).convert('RGB')
fig, ax = plt.subplots(figsize=(10, 7)); ax.imshow(img); ax.axis('off')
cmap = plt.get_cmap('tab10')
for w in rec['words']:
    x1, y1, x2, y2 = w['bbox']
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.5,
                                   edgecolor=cmap(w.get('block', 0) % 10)))
    ax.text(x1, y1, str(w['script'])[:3], color='yellow', fontsize=7, backgroundcolor='black')
ax.set_title(f"{rec['id']} | blocks={rec['n_blocks']} lines={rec['n_lines']} (box colour = block)")
plt.show()